In [ ]:
# Apply FLII mask to benchmarking datasets.
# Run on dedicated CPU node

In [1]:
import rioxarray
import xarray as xr
import geopandas as gpd

In [2]:
mask_path="/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ForestLandscapeIntegrityIndex/flii_6_Amazon_mask_polygon.shp"
# Load shapefile
polygon_gdf = gpd.read_file(mask_path)

In [3]:
def apply_flii_mask(shapefile,filename):

    out_dir="/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/FLII_Masked_Regional_Files/"
    in_dir="/pscratch/sd/j/jkowalcz/e3sm_scratch/pm-cpu/Benchmarks/ILAMB/"

    # Open target xarray dataset
    dataset = xr.open_dataset(in_dir+filename, chunks={'time': 10})

    # Drop any non-spatial time helpers that cause errors
    vars_to_drop = [
        v for v in dataset.variables 
        if ("bnds" in v or "bounds" in v) and v not in ["lat", "lon", "time"]
    ]
    dataset = dataset.drop_vars(vars_to_drop)    

    for var in dataset.variables:
        # Wipe conflicting grid_mapping keys from both attrs and encoding
        if "grid_mapping" in dataset[var].attrs:
            del dataset[var].attrs["grid_mapping"]
        if "grid_mapping" in dataset[var].encoding:
            del dataset[var].encoding["grid_mapping"]
            
        # Wipe of all fill/missing values across all variables
        dataset[var].attrs.pop("_FillValue", None)
        dataset[var].attrs.pop("missing_value", None)
        dataset[var].encoding.pop("_FillValue", None)
        dataset[var].encoding.pop("missing_value", None)
        
    # Ensure the dataset has spatial dimensions specified for rioxarray
    dataset = dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
    dataset = dataset.rio.write_crs("EPSG:4326") # Match your data's CRS

    # Clip the dataset directly using the shapefile geometries
    # This instantly drops everything outside the polygons, saving a ton of memory
    clipped_dataset = dataset.rio.clip(polygon_gdf.geometry, polygon_gdf.crs)
    
    if "time" in clipped_dataset.dims:
        clipped_dataset = clipped_dataset.transpose("time", "lat", "lon", ...)
    else:
        clipped_dataset = clipped_dataset.transpose("lat", "lon", ...)

    # Save masked dataset
    clipped_dataset.to_netcdf(out_dir+filename)

    return

In [27]:
## ILAMB AGB datasets
apply_flii_mask(polygon_gdf,'geocarbon_biomass.nc')
apply_flii_mask(polygon_gdf,'ilamb_biomass_XuSaatchi.nc')
apply_flii_mask(polygon_gdf,'saatchi2011_biomass_0.5x0.5.nc')
apply_flii_mask(polygon_gdf,'esacci_biomass.nc')
apply_flii_mask(polygon_gdf,'CTrees_biomass_Amazon_regional_avg2000-2014.nc')

In [6]:
## ILAMB GPP 
apply_flii_mask(polygon_gdf,'ilamb_fluxcom_gpp.nc')
apply_flii_mask(polygon_gdf,'wecann_gpp.nc')

/global/homes/j/jkowalcz/.conda/envs/xesmf_env/lib/python3.12/site-packages/xarray/core/dataset.py:271: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(


In [15]:
## ILAMB LAI
apply_flii_mask(polygon_gdf,'AVH15C1_lai.nc')
apply_flii_mask(polygon_gdf,'cao2023_lai.nc')
apply_flii_mask(polygon_gdf,'ilamb_modis_lai_0.5x0.5.nc')
apply_flii_mask(polygon_gdf,'avhrr_lai_0.5x0.5.nc')

In [4]:
## ILAMB ET
apply_flii_mask(polygon_gdf,'GLEAMv3.3a_et.nc')
apply_flii_mask(polygon_gdf,'MOD16A2_et.nc')
apply_flii_mask(polygon_gdf,'modis_et_0.5x0.5.nc') 

In [8]:
## ILAMB LH
apply_flii_mask(polygon_gdf,'WECANN_LH.nc')
apply_flii_mask(polygon_gdf,'fluxcom_LH.nc')
apply_flii_mask(polygon_gdf,'CLASS_LH.nc')
apply_flii_mask(polygon_gdf,'DOLCE_LH.nc')

/global/homes/j/jkowalcz/.conda/envs/xesmf_env/lib/python3.12/site-packages/xarray/core/dataset.py:271: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 10. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(
